In [ ]:
TYPE_DESCRIPTIONS = {
    "overview": "queries about which subjects match certain characteristics (e.g., taught in a specific semester, related to a topic, or general overviews of subject goals, credits, syllabus, or curriculum structure.",
    "construtive_question": "thought-provoking questions that encourage critical thinking or reflection.",
    "assessment": "evaluations, types of tests, exams, final exams (FE), progress exams (PE), term exams (TE), and grading weights.",
    "session": "lecture sessions, lessons, and topics covered in each week or session.",
    "material": "recommended textbooks, reference materials, lecture slides, readings, or other learning resources.",
    "learning outcome": "expected knowledge, skills, or competencies students should gain after completing the course.",
    "guide": "instructions or guidance for students on how to complete tasks, assignments, projects, or how to use certain tools or platforms.",
    "student_list": "list of students enrolled in the course, including names, student IDs, and email addresses.",
    "attendance": "records of student attendance per session, including presence or absence, date, room, and instructor.",
    "grade detail": "detailed grade components, including evaluation item name, category, weight, and obtained score.",
    "course summary": "final course summary including average score, status, and summary notes about course performance.",
    "student profile": "personal profile of an unique student, including full name, student ID, email, program, and major."
}

TYPE_KEYWORDS = {
    "overview": ["overview", "objective", "goal", "credits", "semester", "prerequisite", "syllabus", "subject", "subjects", 'general'],
    "construtive_question": ["why", "how", "what if", "critical", "discussion", "reflect", "ethical", "opinion", "thinking"],
    "assessment": ["exam", "test", "quiz", "grading", "project", "evaluation", "score", "mark", "weight", "assessment"],
    "session": ["week", "lesson", "lecture", "topic", "schedule", "session", "class", "timetable"],
    "material": ["textbook", "slide", "document", "reading", "reference", "material", "resource", "book", "pdf", "file"],
    "learning outcome": ["learn", "outcome", "skill", "competency", "ability", "achieve", "knowledge", "CLO", "LO", "learning outcome"],
    "student_list": ["student", "name", "id", "mssv", "email", "class list", "enrolled", "danh sách sinh viên", "học sinh", "danh sách"],
    "guide": ["how to", "instruction", "guide", "tutorial", "step", "steps", "do", "complete", "submit", "platform", "tool", "usage", "help", "assist", "support", "direction"],
    "attendance": ["attendance", "present", "absent", "record", "check-in", "participation", "presence", "ca học", "phòng", "giảng viên"],
    "grade detail": ["score", "mark", "value", "grade", "item", "category", "evaluation", "component", "trọng số", "điểm"],
    "course summary": ["summarize","summary", "average", "final", "result", "status", "performance", "overall", "total", "điểm trung bình", "kết quả"],
    "student profile": ["student", "profile", "id", "name", "email", "program", "major", "class", "course", "personal"]
}


In [1]:
!pip install transformers sentence-transformers torch


In [16]:
from datasets import load_dataset
from transformers import DistilBertTokenizerFast

# Load file jsonl
dataset = load_dataset("json", data_files={"train": "D:\Learn\Semester_5\SEG301\DEMO_local\FLM\intent.jsonl"})["train"]

# Build label mapping
label_list = sorted(set(dataset["label"]))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

# Map string label -> int và lưu thành "label"
dataset = dataset.map(lambda x: {"label": int(label2id[x["label"]])})

# Tokenize
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
dataset = dataset.map(lambda x: tokenizer(x["text"], truncation=True, padding=True), batched=True)

# Remove old columns
dataset = dataset.remove_columns(["text"])  # nếu không sẽ bị lỗi khi Trainer gọi collate


<>:5: SyntaxWarning: invalid escape sequence '\L'
<>:5: SyntaxWarning: invalid escape sequence '\L'
C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_33844\2661758780.py:5: SyntaxWarning: invalid escape sequence '\L'
  dataset = load_dataset("json", data_files={"train": "D:\Learn\Semester_5\SEG301\DEMO_local\FLM\intent.jsonl"})["train"]


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

c:\Users\DO TUAN MINH\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/240 [00:00<?, ? examples/s]

In [18]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./intent_model",
    per_device_train_batch_size=8,
    num_train_epochs=10,
    save_steps=20,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer
)

trainer.train()
trainer.save_model("./intent_model")


c:\Users\DO TUAN MINH\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/300 [00:00<?, ?it/s]

{'train_runtime': 32.2033, 'train_samples_per_second': 74.527, 'train_steps_per_second': 9.316, 'train_loss': 0.7262137349446615, 'epoch': 10.0}


In [2]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import torch

# Load mô hình và tokenizer đã fine-tune
model = DistilBertForSequenceClassification.from_pretrained("./intent_model")
tokenizer = DistilBertTokenizerFast.from_pretrained("./intent_model")
model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
import torch
import requests
def predict_type(query_en: str) -> str:
    inputs = tokenizer(query_en, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=1).item()
    return model.config.id2label[pred]

# === 2. Dịch bằng Gemini ===
def translate_with_gemini(text: str, api_key: str, model: str = "models/gemini-2.0-flash") -> str:
    url = f"https://generativelanguage.googleapis.com/v1beta/{model}:generateContent?key={api_key}"
    headers = {"Content-Type": "application/json"}
    prompt = f", just answer shortly, Translate this Vietnamese query to English:\n{text}"
    data = {"contents": [{"parts": [{"text": prompt}]}]}
    try:
        res = requests.post(url, headers=headers, json=data)
        res.raise_for_status()
        return res.json()['candidates'][0]['content']['parts'][0]['text'].strip()
    except Exception as e:
        print(f"Lỗi dịch Gemini: {e}")
        return "[Translation failed]"

# === 3. Test ===
api_key = ""  # <-- Thay bằng Gemini API key thật

test_queries_vi = [
    'Tổng hợp về môn CPV',
    'Điểm môn lab CPV của tôi',
    'Điểm danh tuần trước',
    'Học MAI thì đọc sách gì',
    'hệ số FE môn DPL',
    'các môn kì 6'
]

for q in test_queries_vi:
    print(f"🇻🇳 Câu gốc: {q}")
    query_en = translate_with_gemini(q, api_key)
    print(f"🇺🇸 Dịch: {query_en}")
    intent = predict_type(query_en)
    print(f"📌 Intent dự đoán: {intent}")
    print("-" * 60)


🇻🇳 Câu gốc: Tổng hợp về môn CPV
🇺🇸 Dịch: Summary about the subject CPV.
📌 Intent dự đoán: course summary
------------------------------------------------------------
🇻🇳 Câu gốc: Điểm môn lab CPV của tôi
🇺🇸 Dịch: My CPV lab score/grade.
📌 Intent dự đoán: grade detail
------------------------------------------------------------
🇻🇳 Câu gốc: Điểm danh tuần trước
🇺🇸 Dịch: Attendance last week.
📌 Intent dự đoán: attendance
------------------------------------------------------------
🇻🇳 Câu gốc: Học MAI thì đọc sách gì
🇺🇸 Dịch: What books to read for MAI?
📌 Intent dự đoán: material
------------------------------------------------------------
🇻🇳 Câu gốc: hệ số FE môn DPL
🇺🇸 Dịch: FE coefficient in the DPL subject
📌 Intent dự đoán: assessment
------------------------------------------------------------
🇻🇳 Câu gốc: các môn kì 6
🇺🇸 Dịch: Subjects in semester 6
📌 Intent dự đoán: overview
------------------------------------------------------------


In [22]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L6-v2")

def detect_subjects(query_en: str, subject_map: dict, threshold=0.6):
    subject_texts = [f"{code} {name}" for code, name in subject_map.items()]
    query_emb = embedder.encode(query_en, convert_to_tensor=True)
    subject_embs = embedder.encode(subject_texts, convert_to_tensor=True)

    cos_scores = util.cos_sim(query_emb, subject_embs)[0]
    top_idxs = torch.where(cos_scores > threshold)[0]
    matched_codes = [list(subject_map.keys())[i] for i in top_idxs]
    return matched_codes

import re

def extract_semester(query: str) -> int | None:
    patterns = [
        r"k[ỳì]\s*(\d+)",      # kỳ 5
        r"semester\s*(\d+)",   # semester 3
        r"term\s*(\d+)",       # term 2
        r"k[ỳì]\s*cuối",       # kỳ cuối → có thể gán là 9
        r"k[ỳì]\s*đ[âầu]",     # kỳ đầu → 0
    ]
    
    for p in patterns:
        match = re.search(p, query.lower())
        if match:
            try:
                return int(match.group(1))
            except:
                if "cuối" in p: return 9
                if "đầu" in p: return 0
    return None


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\DO TUAN MINH\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DO TUAN MINH\.cache\huggingface\hub\models--sentence-transformers--paraphrase-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [31]:
subject_map

{'OTP101': 'OTP101 - Orientation and General Training Program_Định hướng và Rèn luyện tập trung',
 'PEN': 'PEN - Preparation English_Tiếng Anh chuẩn bị',
 'PHE_COM*1': 'PHE_COM*1 - Physical Education 1_Giáo dục thể chất 1',
 'VOV114': 'VOV114 - Vovinam 1',
 'COV111': 'COV111 - Cờ Vua 1_Chess 1',
 'TMI_ELE': 'TMI_ELE - Traditional musical instrument_Nhạc cụ truyền thống',
 'DTR103': 'DTR103 - Nhạc cụ truyền thống-Đàn Tranh',
 'DBA103': 'DBA103 - Nhạc cụ truyền thống - Đàn Bầu',
 'DSA103': 'DSA103 - Nhạc cụ truyền thống- Sáo trúc',
 'DNG103': 'DNG103 - Nhạc cụ truyền thống-Đàn nguyệt',
 'DTB103': 'DTB103 - Nhạc cụ truyền thống- Đàn Tỳ bà',
 'TRG103': 'TRG103 - Nhạc cụ truyền thống -Trống dân tộc',
 'DNH103': 'DNH103 - Nhạc cụ truyền thống- Đàn Nhị',
 'CSI106': 'CSI106 - Introduction to Computer Science_Nhập môn khoa học máy tính',
 'MAD101': 'MAD101 - Discrete mathematics_Toán rời rạc',
 'MAE101': 'MAE101 - Mathematics for Engineering_Toán cho ngành kỹ thuật',
 'PFP191': 'PFP191 - Progr

In [30]:
def analyze_intent(query_vi: str, subject_map: dict, api_key: str) -> dict:
    query_en = translate_with_gemini(query_vi, api_key)
    intent = predict_type(query_en)
    subjects = detect_subjects(query_en, subject_map)
    semester = extract_semester(query_vi)

    result = {
        "type": intent,
        "subjects": subjects,
        "query_en": query_en
    }
    if semester is not None:
        result["semester"] = semester

    return result

analyze_intent('môn nào liên quan tới xử lí ảnh', subject_map=subject_map, api_key=api_key)

{'type': 'overview',
 'subjects': [],
 'query_en': 'Which subjects are related to image processing?'}